In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf 
tf.config.set_visible_devices([], "GPU")
import keras
from preprocessing import train_ds, valid_ds, combined_loss, dice_metric, TRAIN_STEPS

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(3e-4)

inputs=keras.layers.Input(shape=(288,288,3))

# Encoder which downscales the image
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(inputs)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
block_1_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_1=keras.layers.Conv2D(filters=64, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_1])
block_2_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_2=keras.layers.Conv2D(filters=128, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_2])
block_3_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_3=keras.layers.Conv2D(filters=256, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_3])
block_4_output=x
x=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

bottleneck=x

# Decoder which upscales the images

x=keras.layers.Conv2DTranspose(filters=256,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(bottleneck)
x=keras.layers.Concatenate()([x,block_4_output])
x=keras.layers.Dropout(0.2)(x)

block_1_decoder_output=keras.layers.Conv2D(256, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_1_decoder_output])

x=keras.layers.Conv2DTranspose(filters=128,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_3_output])
x=keras.layers.Dropout(0.2)(x)

block_2_decoder_output=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_2_decoder_output])

x=keras.layers.Conv2DTranspose(filters=64,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_2_output])
x=keras.layers.Dropout(0.2)(x)

block_3_decoder_output=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_3_decoder_output])

x=keras.layers.Conv2DTranspose(filters=32,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_1_output])
x=keras.layers.Dropout(0.2)(x)

block_4_decoder_output=keras.layers.Conv2D(32, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_4_decoder_output])

outputs=keras.layers.Conv2D(1, kernel_size=(1,1), activation='sigmoid')(x)

model=keras.Model(inputs, outputs)

adamW=keras.optimizers.AdamW(3e-4, clipnorm=1.0, weight_decay=1e-4)

sgd=keras.optimizers.SGD(1e-3,momentum=0.95, nesterov=True)

model.compile(
    optimizer=adamW,
    loss=combined_loss,
    metrics=[dice_metric]
)

earlyStop_cb=keras.callbacks.EarlyStopping(
    monitor='val_dice_metric',
    patience=10,
    verbose=1,
    restore_best_weights=True,
    mode='max'
)

lrPlateau_cb=keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_metric',
    patience=3,
    factor=0.5,
    verbose=1,
    min_lr=5e-7,
    mode='max'
)

history=model.fit(
    train_ds,
    batch_size=32,
    epochs=40,
    callbacks=[earlyStop_cb, lrPlateau_cb],
    validation_data=valid_ds,
    steps_per_epoch=TRAIN_STEPS
)

model.save("Saved Models/Seg_ResidualUNetStyle_adamw_lr3e-4_combinedloss_dicemetric.keras")

Epoch 1/40


I0000 00:00:1777570138.319630 1672439 service.cc:145] XLA service 0x9dcc7a200 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777570138.319647 1672439 service.cc:153]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1777570138.459232 1672439 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - dice_metric: 0.1534 - loss: 8.3712 - val_dice_metric: 0.1526 - val_loss: 10.1640 - learning_rate: 3.0000e-04
Epoch 2/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - dice_metric: 0.1494 - loss: 7.1777 - val_dice_metric: 0.1490 - val_loss: 8.6075 - learning_rate: 3.0000e-04
Epoch 3/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - dice_metric: 0.1501 - loss: 6.5435 - val_dice_metric: 0.1533 - val_loss: 7.2624 - learning_rate: 3.0000e-04
Epoch 4/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - dice_metric: 0.1534 - loss: 6.0711 - val_dice_metric: 0.1511 - val_loss: 6.6590 - learning_rate: 3.0000e-04
Epoch 5/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - dice_metric: 0.1464 - loss: 5.7272 - val_dice_metric: 0.1452 - val_loss: 6.1446 - learning_rate: 3.0000e-04
Epoch 6/40
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - dice_metric: 0.1445 - loss: 5.5192
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - dice_metric: 0.1445 - los

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf 
tf.config.set_visible_devices([], "GPU")
import keras
from preprocessing import train_ds, valid_ds, combined_loss, dice_metric

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(3e-4)

inputs=keras.layers.Input(shape=(288,288,3))

# Encoder which downscales the image
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(inputs)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
block_1_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_1=keras.layers.Conv2D(filters=64, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_1])
block_2_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_2=keras.layers.Conv2D(filters=128, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_2])
block_3_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_3=keras.layers.Conv2D(filters=256, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_3])
block_4_output=x
x=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

bottleneck=x

# Decoder which upscales the images

x=keras.layers.Conv2DTranspose(filters=256,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(bottleneck)
x=keras.layers.Concatenate()([x,block_4_output])
x=keras.layers.Dropout(0.2)(x)

block_1_decoder_output=keras.layers.Conv2D(256, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_1_decoder_output])

x=keras.layers.Conv2DTranspose(filters=128,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_3_output])
x=keras.layers.Dropout(0.2)(x)

block_2_decoder_output=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_2_decoder_output])

x=keras.layers.Conv2DTranspose(filters=64,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_2_output])
x=keras.layers.Dropout(0.2)(x)

block_3_decoder_output=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_3_decoder_output])

x=keras.layers.Conv2DTranspose(filters=32,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_1_output])
x=keras.layers.Dropout(0.2)(x)

block_4_decoder_output=keras.layers.Conv2D(32, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_4_decoder_output])

outputs=keras.layers.Conv2D(1, kernel_size=(1,1), activation='sigmoid')(x)

model=keras.Model(inputs, outputs)

adamW=keras.optimizers.AdamW(3e-4, clipnorm=1.0, weight_decay=1e-4)

sgd=keras.optimizers.SGD(1e-3,momentum=0.95, nesterov=True)

model.compile(
    optimizer=adamW,
    loss=combined_loss,
    metrics=[dice_metric]
)

earlyStop_cb=keras.callbacks.EarlyStopping(
    monitor='val_dice_metric',
    patience=10,
    verbose=1,
    restore_best_weights=True,
    mode='max'
)

lrPlateau_cb=keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_metric',
    patience=3,
    factor=0.5,
    verbose=1,
    min_lr=5e-7,
    mode='max'
)

history=model.fit(
    train_ds,
    batch_size=32,
    epochs=40,
    callbacks=[earlyStop_cb, lrPlateau_cb],
    validation_data=valid_ds,
    steps_per_epoch=100
)

model.save("Saved Models/Seg_ResidualUNetStyle_adamw_lr3e-4_combinedloss_dicemetric.keras")

Epoch 1/40
100/100 ━━━━━━━━━━━━━━━━━━━━ 752s 7s/step - dice_metric: 0.5266 - loss: 3.8929 - val_dice_metric: 0.0398 - val_loss: 4.3413 - learning_rate: 3.0000e-04
Epoch 2/40
100/100 ━━━━━━━━━━━━━━━━━━━━ 795s 8s/step - dice_metric: 0.7217 - loss: 2.7348 - val_dice_metric: 0.1515 - val_loss: 3.6377 - learning_rate: 3.0000e-04
Epoch 3/40
100/100 ━━━━━━━━━━━━━━━━━━━━ 694s 7s/step - dice_metric: 0.7502 - loss: 2.2199 - val_dice_metric: 0.3749 - val_loss: 2.8188 - learning_rate: 3.0000e-04
Epoch 4/40
100/100 ━━━━━━━━━━━━━━━━━━━━ 648s 6s/step - dice_metric: 0.7708 - loss: 1.8742 - val_dice_metric: 0.6512 - val_loss: 1.9985 - learning_rate: 3.0000e-04
Epoch 5/40
100/100 ━━━━━━━━━━━━━━━━━━━━ 654s 7s/step - dice_metric: 0.7882 - loss: 1.6314 - val_dice_metric: 0.7137 - val_loss: 1.6990 - learning_rate: 3.0000e-04
Epoch 6/40
100/100 ━━━━━━━━━━━━━━━━━━━━ 634s 6s/step - dice_metric: 0.8048 - loss: 1.4468 - val_dice_metric: 0.7570 - val_loss: 1.4743 - learning_rate: 3.0000e-04
Epoch 7/40
100/100 ━━━